# Week 99, Day 3: Python deep-dive interview

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w099d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __eq__(self, other):
        return (self.x, self.y) == (other.x, other.y)

print(Point(1, 2) == Point(1, 2), Point.__hash__)

> **Key idea.** Equal objects must have equal hashes, and hashes mustn't change: hashable means effectively immutable; `__eq__` alone makes a class unhashable. Generators are lazy and single-use. Descriptors (`__get__`, `__set__`, `__set_name__`) power `property` and methods. `__exit__` always runs. CPython frees objects at refcount zero and collects cycles with `gc`. The GIL lets one thread run bytecode at a time: threads for waiting, processes for computing; free-threading is experimental since 3.13.

### Exercise: A point you can put in a set

Write a class `Point(x, y)` that compares equal by coordinates, hashes consistently with that, sorts by `(x, y)` (`__lt__`), shows as `Point(1, 2)`, and refuses changes: setting any attribute after creation raises `AttributeError`.

In [ ]:
class Point:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_point():\n    \"equal, hashable, frozen\"\n    a, b = Point(1, 2), Point(1, 2)\n    assert a == b and hash(a) == hash(b) and len({a, b, Point(2, 1)}) == 2\n    assert {a: \"Mill\"}[b] == \"Mill\" and repr(a) == \"Point(1, 2)\"\n    assert sorted([Point(2, 0), Point(1, 5), Point(1, 2)]) == [Point(1, 2), Point(1, 5), Point(2, 0)]\n    assert (a == \"not a point\") is False\n    try:\n        a.x = 9\n    except AttributeError:\n        pass\n    else:\n        raise AssertionError(\"points are frozen\")")

### Exercise: Stopping early

Write a generator `long_walks(lines, min_km)` yielding the km (floats) of lines `"date,km"` with at least `min_km`, and `first_long(lines, min_km, n)` returning a list of the first `n` of them by taking from the generator. Given an endless source, it must stop reading as soon as it has `n`.

In [ ]:
from itertools import islice

def long_walks(lines, min_km):
    ___

def first_long(lines, min_km, n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_lazy():\n    \"read only what's needed\"\n    read = []\n    def source():\n        i = 0\n        while True:\n            i += 1\n            read.append(i)\n            yield f\"2026-01-{i % 28 + 1:02d},{i % 12}.0\"\n    assert first_long(source(), 10, 2) == [10.0, 11.0] and len(read) == 11\n    assert list(long_walks([\"d,3.5\", \"d,12.0\"], 5)) == [12.0]")

### Exercise: A validating attribute

Write a descriptor class `Positive` so that `km = Positive()` on a class makes `km` an attribute that must be a positive number: setting anything else raises `ValueError("km must be positive")` (using the attribute's own name, from `__set_name__`). Values are stored on the instance under `"_" + name`; reading it from the class returns the descriptor itself.

In [ ]:
class Positive:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_descriptor():\n    \"property's mechanism\"\n    class Walk:\n        km = Positive()\n        minutes = Positive()\n        def __init__(self, km, minutes):\n            self.km, self.minutes = km, minutes\n    w = Walk(6.0, 105)\n    assert w.km == 6.0 and w.minutes == 105 and w._km == 6.0\n    for bad in (0, -1, \"6\"):\n        try:\n            w.km = bad\n        except ValueError as e:\n            assert str(e) == \"km must be positive\"\n        else:\n            raise AssertionError(bad)\n    try:\n        Walk(5, -3)\n    except ValueError as e:\n        assert str(e) == \"minutes must be positive\"\n    assert isinstance(Walk.km, Positive)")

### Exercise: Clean-up, guaranteed

Write a context manager class `temporarily(obj, attr, value)`: on entry it sets `obj.attr` to `value` (remembering the old value), and on exit it restores the old value, even if the block raised; it must not swallow exceptions.

In [ ]:
class temporarily:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_context():\n    \"__exit__ always runs\"\n    class Settings:\n        steep_limit = 80\n    s = Settings()\n    with temporarily(s, \"steep_limit\", 120):\n        assert s.steep_limit == 120\n    assert s.steep_limit == 80\n    try:\n        with temporarily(s, \"steep_limit\", 200):\n            raise KeyError(\"boom\")\n    except KeyError:\n        pass\n    else:\n        raise AssertionError(\"not swallowed\")\n    assert s.steep_limit == 80")

### Exercise: Watching a cycle disappear

Write `cycle_is_collected()`: make two objects of a small class that refer to each other, keep only a `weakref.ref` to one, delete your names for them, call `gc.collect()`, and return whether the weak reference is now dead (`ref() is None`).

In [ ]:
import gc
import weakref

def cycle_is_collected():
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cycle():\n    \"the cycle collector\"\n    assert cycle_is_collected() is True")

### Bug hunt: Equal, but not hashable

The class defines `__eq__` but not `__hash__`, so putting junctions in a set raises `TypeError: unhashable type`. Add a `__hash__` consistent with `__eq__` so this prints `2`.

In [ ]:
class Junction:
    def __init__(self, name):
        self.name = name
    def __eq__(self, other):
        return isinstance(other, Junction) and self.name == other.name

print(len({Junction("Mill"), Junction("Mill"), Junction("Weir")}))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"equal objects, equal hashes\"\n    expect_output(\"2\")")

### Bug hunt: A total that vanished

A generator can only be used once, so the second `sum` sees nothing and this prints `10 0`. Keep the values in a list so it prints `10 10`.

In [ ]:
km = (x for x in [1, 2, 3, 4])
print(sum(km), sum(km))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"single-use\"\n    expect_output(\"10 10\")")

### Puzzle: A descriptor's get

Arrange `__get__` so reading from the class gives the descriptor and reading from an instance gives the stored value. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
class A:
    def who(self):
        return "A"
class B(A):
    def who(self):
        return "B" + super().who()
class C(A):
    def who(self):
        return "C" + super().who()
class D(B, C):
    pass

p = ___   # D().who()
q = ___   # [k.__name__ for k in D.__mro__][:4]
r = ___   # True or False: threads speed up pure-Python computation in standard CPython
s = ___   # what defining __eq__ alone sets __hash__ to
t = ___   # the module that collects reference cycles

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"BCA\" and q == [\"D\", \"B\", \"C\", \"A\"] and r is False, \"p, q and r\"\n    assert s is None and t == \"gc\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: your own lru_cache

Write a decorator factory `memo(maxsize)`: the decorated function caches results by its positional arguments, keeps at most `maxsize` of them (evicting the least recently used), keeps its `__name__` and docstring (`functools.wraps`), and gains `cache_info()` returning `(hits, misses, size)` and `cache_clear()`.

In [ ]:
import functools
from collections import OrderedDict

def memo(maxsize):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_memo():\n    \"hits, misses, eviction\"\n    calls = []\n    @memo(maxsize=2)\n    def climb(a, b):\n        \"\"\"Climb between two junctions.\"\"\"\n        calls.append((a, b))\n        return abs(a - b)\n    assert climb(40, 310) == 270 and climb(40, 310) == 270 and climb.cache_info() == (1, 1, 1)\n    climb(55, 80)\n    climb(40, 310)\n    climb(1, 2)\n    assert climb.cache_info() == (2, 3, 2)\n    climb(55, 80)\n    assert calls.count((55, 80)) == 2, \"the least recently used was evicted\"\n    assert climb.__name__ == \"climb\" and climb.__doc__ == \"Climb between two junctions.\"\n    climb.cache_clear()\n    assert climb.cache_info() == (0, 0, 0)\n\ndef test_recursive():\n    \"memoised recursion\"\n    @memo(maxsize=1000)\n    def ways(n):\n        return 1 if n < 2 else ways(n - 1) + ways(n - 2)\n    assert ways(80) == 37889062373143906")

## Warm-up

Defining `__eq__` without `__hash__` makes the class **unhashable** (`__hash__` becomes `None`), because objects that are equal must have equal hashes, and Python can't guess one. That's the answer to "why can't a list be a key?" too: lists compare by contents, which can change, so a hash based on them would go stale.

## Objects and hashing

A dictionary or set finds a key by its **hash**, then confirms with `==`. The rule: **if `a == b`, then `hash(a) == hash(b)`**, and an object's hash must not change while it's in a set or dictionary. So hashable objects are usually **immutable**: tuples, strings, frozen dataclasses. Make your own by defining `__eq__` and `__hash__` from the same fields and refusing changes (or with `@dataclass(frozen=True)`).

## Laziness, descriptors, context managers

- **Generators** produce values on demand: a pipeline of generators reads a 10 GB log one line at a time, and stops reading as soon as the consumer stops asking. They can be used **once**: a second loop over an exhausted generator sees nothing.
- **Descriptors** are objects stored on a class that control attribute access on its instances through `__get__`, `__set__` and `__set_name__`. `property`, methods themselves and `classmethod` are descriptors; a validating attribute (`km = Positive()`) is the classic example.
- **Context managers** guarantee clean-up: `__exit__` runs however the block ends, with the exception (if any) as its arguments; returning a true value would swallow it, which is rarely wanted.

## Memory and threads

CPython frees most objects the moment their **reference count** reaches zero. Objects that refer to each other in a **cycle** never reach zero on their own, so a separate **cycle collector** (`gc`) finds and frees unreachable groups from time to time. `weakref` lets you refer to an object without keeping it alive, which is how you can watch it go.

The **GIL** (global interpreter lock) lets only one thread run Python bytecode at a time in standard CPython, which keeps reference counting simple and safe. So threads help with **waiting** (network, files) but not with pure-Python **computation**; for that use processes (`multiprocessing`, `concurrent.futures.ProcessPoolExecutor`) or code that releases the GIL (NumPy). Python 3.13 added an experimental **free-threaded** build without the GIL (PEP 703); say that it exists and is still experimental, not that the GIL is gone.

## Practice

## Quiz

## Daily challenge

## A mock round

Ask a friend to quiz you for twenty minutes from this list, and answer each with a small example, typed or on paper: why `{}` can't hold lists; what `yield` does to a function; how `@property` works; what `with open(...)` guarantees; what the GIL does and doesn't do; when `gc` matters; what `super()` follows. Note the ones where you needed the example to remember.

## Before you go

Tomorrow and the day after are full mock loops, each mixing everything from the last three weeks.